In [11]:
import pandas as pd
from app.analysis.matching import load_analysis_data
from app.db.session import engine

df_rooms = pd.read_sql("SELECT * FROM rooms WHERE building_ID = 214", engine)
print("=== rooms table ===")
print(df_rooms)

df_buildings = pd.read_sql("SELECT * FROM buildings LIMIT 2", engine)
print("=== buildings table ===")
print(df_buildings)    

=== rooms table ===
     id  building_id     suumo_room_id     price  admin_fee  monthly_fee  \
0   670          214  jnc_000108889621       NaN        NaN          NaN   
1   662          214  jnc_000107781937  174000.0     8000.0     182000.0   
2   663          214  jnc_000107781941  178000.0     8000.0     186000.0   
3   664          214  jnc_000107798181  179000.0     8000.0     187000.0   
4   665          214  jnc_000107781938  197000.0    10000.0     207000.0   
5   666          214  jnc_000107781946  201000.0    10000.0     211000.0   
6   667          214  jnc_000108942819  209000.0    10000.0     219000.0   
7   668          214  jnc_000108882165  209000.0    10000.0     219000.0   
8   669          214  jnc_000108882166  213000.0    10000.0     223000.0   
9   671          214  jnc_000108882167  214000.0    10000.0     224000.0   
10  672          214  jnc_000107781936  215000.0    10000.0     225000.0   
11  673          214  jnc_000107767898  215000.0    10000.0     2250

In [10]:
import pandas as pd

# df_raw は DBにUpsertする前の、Suumoからスクレイピングした生のDataFrameを想定
# カラム: title, address, age (建築年), total_floors, station_distance, price など

def detect_attribute_conflicts(df_raw: pd.DataFrame):
    # title と address でグループ化し、各属性のユニーク（一意）な値の数をカウント
    variance_df = df_raw.groupby(["title", "address"]).agg(
        record_count=("title", "count"),
        unique_ages=("age", "nunique"),
        unique_total_floors=("total_floors", "nunique"),
        unique_stations=("station_distance", "nunique")
    ).reset_index()

    # 建築年、総階数、駅徒歩のいずれかが「2種類以上」混在しているグループを抽出
    suspicious = variance_df[
        (variance_df["unique_ages"] > 1) |
        (variance_df["unique_total_floors"] > 1) |
        (variance_df["unique_stations"] > 1)
    ]
    
    return suspicious.sort_values("record_count", ascending=False)

# 実行と確認
suspicious_buildings = detect_attribute_conflicts(df_buildings)
display(suspicious_buildings.head(10))

,title,address,record_count,unique_ages,unique_total_floors,unique_stations


In [3]:
import os
import sys

# プロジェクトルート (/app) を Python の検索パスに追加
sys.path.append(os.path.abspath(".."))

import pandas as pd
from app.analysis.matching import load_analysis_data
from app.db.session import engine

# DBから現状の分析用データを読み込み
df = load_analysis_data(engine)

def detect_overcrowded_floors(df: pd.DataFrame, threshold: int = 6):
    """
    1つの階に不自然に多くの部屋がある建物を検知する。
    タワマンや大型マンションの誤検知を防ぐため、総階数が低い建物を対象にする。
    """
    # 3階建て以下の低層物件（アパート等）に限定
    df_low_rise = df[df["total_floors"] <= 3]
    
    # 建物IDと所在階ごとに部屋数をカウント
    floor_crowding = df_low_rise.groupby(["building_id", "title", "floor"]).agg(
        room_count=("room_id", "count")
    ).reset_index()
    
    # 1フロアに指定数(threshold)以上の部屋があるデータを抽出
    suspicious = floor_crowding[floor_crowding["room_count"] >= threshold]
    
    return suspicious.sort_values("room_count", ascending=False)

# 実行と確認（例: 3階建て以下なのに1フロアに6部屋以上ある物件）
overcrowded_df = detect_overcrowded_floors(df, threshold=6)
display(overcrowded_df.head(10))

,building_id,title,floor,room_count
48,214,ウェルスクエア幡ヶ谷,1.0,6
